# RSNA Knee Abnormality Detection — Weak-Label Training

Does training on report-derived labels from thousands of studies beat training
on 58 human-labelled ones?

An earlier evaluation measured these labels for **precision** and returned a
no-go: none of the twelve cleared a Wilson lower-bound gate, because per-label
support among the 58 human labels is only about ten to twenty. That answered
whether an individual weak label is reliable enough to stand on its own. It did
not answer whether training on them helps, which is a different question with a
cleaner test: **fit on report-only studies, evaluate on the human-labelled
ones.** Noisy training labels cannot corrupt a human-labelled evaluation set.

The reports are a training-time asset only. The competition test set carries no
report column, and the model here stays image-only at prediction time, so that
absence does not constrain this.

## 1. Environment and Frozen Configuration

In [ ]:
import hashlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

# Verify and install the pinned offline wheel BEFORE inserting the source
# path or importing knee_mri anywhere, so the import cannot silently pick up
# a different stratifier than the one this contract pins.
package_initializers = tuple(Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py"))
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
_src_root = package_initializers[0].parent.parent
_dataset_root = _src_root.parent

WHEEL_NAME = "iterative_stratification-0.1.9-py3-none-any.whl"
EXPECTED_SHA256 = "476f8deff6753fb1725612fe41e59cc2058f8f2524ae5d1ccee88eb8c8d3de80"

wheel_matches = tuple(_dataset_root.rglob(WHEEL_NAME))
if len(wheel_matches) != 1:
    raise RuntimeError("Expected exactly one pinned iterative-stratification wheel.")
wheel_path = wheel_matches[0]
if hashlib.sha256(wheel_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Pinned iterative-stratification wheel checksum mismatch.")

try:
    install_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", str(wheel_path)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the pinned wheel.") from None
if install_result.returncode != 0:
    raise RuntimeError("Offline installation of the pinned wheel failed.")
if importlib.metadata.version("iterative-stratification") != "0.1.9":
    raise RuntimeError("Installed iterative-stratification version mismatch.")

# The processor statistics come from the vendored copy of the attached
# model's own preprocessor_config.json. There is deliberately no fallback:
# substituting remembered constants is the silent-wrongness this contract
# exists to prevent.
PROCESSOR_CONFIG_NAME = "dinov2-small-preprocessor_config.json"
processor_matches = tuple(_dataset_root.rglob(PROCESSOR_CONFIG_NAME))
if len(processor_matches) != 1:
    raise RuntimeError("Expected exactly one vendored DINOv2 processor config.")
PROCESSOR_CONFIG_PATH = processor_matches[0]

# Install the vendored DICOM codec plugins. The corpus-wide census found no
# compressed series in either released split, so these are insurance for the
# hidden set rather than a current requirement -- but an undecodable slice
# there would fail silently into the fallback row, which is the failure mode
# worth spending a few seconds to avoid.
#
# --no-deps is required, not stylistic: both compiled wheels declare
# numpy>=2.0,<3.0, and without it pip would try to resolve or replace the
# kernel's own numpy, offline and unasked.
CODEC_WHEELS = {
    "pylibjpeg-2.1.0-py3-none-any.whl":
        "25df9496a69e64e98c887fddee12a1271e275b5f74ba804f9bf98a08bb80993e",
    "pylibjpeg_openjpeg-2.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "a22fcb649ba9849209d8e43dba88632445a5941f0cd6765338b3652a4c686140",
    "pylibjpeg_libjpeg-2.4.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl":
        "01d950ef496476a9223e4966376cb88098fcf5c55a12a21f722d7b5f84daae43",
}

codec_paths = []
for codec_name, codec_sha256 in CODEC_WHEELS.items():
    matches = tuple(_dataset_root.rglob(codec_name))
    if len(matches) != 1:
        raise RuntimeError("Expected exactly one copy of each vendored codec wheel.")
    if hashlib.sha256(matches[0].read_bytes()).hexdigest() != codec_sha256:
        raise RuntimeError("Vendored codec wheel checksum mismatch.")
    codec_paths.append(str(matches[0]))

try:
    codec_result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *codec_paths],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
except OSError:
    raise RuntimeError("Failed to launch the offline install for the codec wheels.") from None
if codec_result.returncode != 0:
    raise RuntimeError("Offline installation of the codec wheels failed.")

sys.path.insert(0, str(_src_root))

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

In [ ]:
import torch
from transformers import AutoModel

from knee_mri.dataset import split_labeled_studies, validated_plane_candidates
from knee_mri.image_model import (
    CONTINUOUS_DIMENSIONS,
    IMAGE_CLASSIFIER_C,
    build_image_classifier,
    cross_validate_image_model,
    fold_signature,
    paired_bootstrap_delta,
)
from knee_mri.intensity import load_processor_statistics
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.laterality import DOMINANCE_GATE, SeriesLateralityEvidence, study_laterality
from knee_mri.metrics import macro_auc
from knee_mri.model_selection import select_multilabel_folds
from knee_mri.series_audit import audit_series
from knee_mri.slice_sampling import (
    MINIMUM_DECODED_SLICES,
    SLICE_SAMPLE_SIZE,
    select_plane_sample,
)
from knee_mri.study_features import (
    EMBEDDING_DIM,
    PLANES,
    STUDY_VECTOR_DIM,
    PlaneInput,
    build_study_features,
    mean_pool,
)
from knee_mri.weak_supervision import fit_weak_label_heads, weak_label_frame

In [ ]:
IMAGE_MEAN, IMAGE_STD = load_processor_statistics(PROCESSOR_CONFIG_PATH)

frozen_contract = pd.Series(
    {
        "Study vector dimensions": STUDY_VECTOR_DIM,
        "Embedding dimensions": EMBEDDING_DIM,
        "Presence + reliability flags": STUDY_VECTOR_DIM - EMBEDDING_DIM,
        "Slices sampled per plane": SLICE_SAMPLE_SIZE,
        "Minimum decoded slices per plane": MINIMUM_DECODED_SLICES,
        "Laterality dominance gate": DOMINANCE_GATE,
        "Classifier C": IMAGE_CLASSIFIER_C,
        "Classifier penalty": build_image_classifier().estimator.penalty,
        "Classifier solver": build_image_classifier().estimator.solver,
        "Classifier class_weight": build_image_classifier().estimator.class_weight,
        "Classifier max_iter": build_image_classifier().estimator.max_iter,
        "Fold candidates": "(5, 4, 3, 2)",
        "Fold seed": SEED,
        "Scaled dimensions (flags unscaled)": CONTINUOUS_DIMENSIONS,
        "Processor image_mean": str(IMAGE_MEAN),
        "Processor image_std": str(IMAGE_STD),
    },
    name="Value",
).to_frame()

display(frozen_contract)

**Interpretation:** the feature configuration is the reported baseline's, unchanged.
This experiment moves the training set, so holding the representation fixed is
what keeps the data lever the only variable.

## 2. Frozen Encoder

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the image baseline.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
_dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(_dinov2_dir), local_files_only=True)
dinov2 = dinov2.to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()
if not GPU_COMPATIBLE:
    raise RuntimeError("Allocated GPU compute capability unsupported by installed PyTorch.")


# Returns the CLS token and the mean of the patch tokens side by side, from a
# single forward pass. The reported baseline uses the CLS half; the
# pre-registered patch-pooling variant uses the other. Computing both here
# rather than in two passes is what guarantees they differ only in the
# representation and in nothing upstream of it.
WIDE_EMBEDDING_DIM = 2 * EMBEDDING_DIM


def encode_batch(batch: torch.Tensor) -> torch.Tensor:
    with torch.no_grad():
        outputs = dinov2(pixel_values=batch.to(DEVICE), interpolate_pos_encoding=True)
    hidden = outputs.last_hidden_state
    cls_token = hidden[:, 0, :]
    patch_mean = hidden[:, 1:, :].mean(dim=1)
    return torch.cat([cls_token, patch_mean], dim=1).detach().cpu()


# Smoke test: a checksum proves the bytes, not that the plugin loads.
codec_plugins = {
    name: importlib.util.find_spec(name) is not None
    for name in ("pylibjpeg", "libjpeg", "openjpeg")
}
if not all(codec_plugins.values()):
    raise RuntimeError("A vendored codec plugin failed to import after install.")

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
        "Codec plugins importable": str(sorted(codec_plugins)),
        "Encoder trainable parameters": sum(
            p.numel() for p in dinov2.parameters() if p.requires_grad
        ),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** records the runtime, and fails here rather than part-way
through a multi-hour extraction if the allocated accelerator is unusable.

## 3. Evaluation Set — the 58 human-labelled studies

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

labeled_studies, unlabeled_studies = split_labeled_studies(train_df)
labeled_studies = labeled_studies.reset_index(drop=True)
unlabeled_studies = unlabeled_studies.reset_index(drop=True)

# Matches the interval budget every other comparison in this project used.
BOOTSTRAP_ITERATIONS = 2_000

# Aggregate-only telemetry. Every entry is a count or a rate; no study or
# series identifier is ever placed in these structures.
telemetry = {
    "planes_absent": 0,
    "plane_retries": 0,
    "candidates_tried": 0,
    "decoded_slice_counts": [],
    "laterality_unreliable_studies": 0,
    "studies_with_no_plane": 0,
    "header_read_failures": 0,
}


def _study_laterality(series_df: pd.DataFrame, series_root: Path, study_id: str, sink=None):
    """Conservative consensus over EVERY available series in the study."""
    sink = telemetry if sink is None else sink
    evidence = []
    study_dir = series_root / study_id
    if not study_dir.is_dir():
        return study_laterality([])
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        try:
            audit = audit_series(series_dir, decode_sample_size=1)
        except FileNotFoundError:
            continue
        sink["header_read_failures"] += audit.header_read_failures
        evidence.append(
            SeriesLateralityEvidence(
                tag=audit.laterality_tag,
                geometry=audit.laterality_from_geometry,
                cross_tag_conflict=audit.laterality_cross_tag_conflict,
            )
        )
    return study_laterality(evidence)


def build_features_for(
    series_df: pd.DataFrame,
    series_root: Path,
    study_ids,
    timings=None,
    per_plane=None,
    sample_size: int = SLICE_SAMPLE_SIZE,
    sink=None,
    slice_pool=mean_pool,
    pooled_width: int = EMBEDDING_DIM,
) -> np.ndarray:
    # A second extraction pass at a different density must not fold its
    # counters into the baseline's, or the reported telemetry becomes a
    # mixture of two contracts. The sink defaults to the baseline's.
    sink = telemetry if sink is None else sink
    vectors = []
    for study_id in study_ids:
        study_start = time.perf_counter()
        study_dir = series_root / study_id
        # Total .dcm files in the study is the real I/O surface: ordering
        # validation reads every header of every candidate series, not just
        # the five slices ultimately decoded.
        study_slices = len(list(study_dir.rglob("*.dcm"))) if study_dir.is_dir() else 0
        planes = {}
        for plane in PLANES:
            candidates = validated_plane_candidates(series_df, series_root, study_id, plane)
            sink["candidates_tried"] += len(candidates)
            outcome = select_plane_sample(
                [paths for _, paths in candidates], sample_size=sample_size
            )
            if outcome.absent or outcome.sample is None:
                sink["planes_absent"] += 1
                continue
            if outcome.candidates_tried > 1:
                sink["plane_retries"] += 1
            sink["decoded_slice_counts"].append(outcome.sample.decoded)
            winning_paths = candidates[outcome.candidates_tried - 1][1]
            header = pydicom.dcmread(winning_paths[0], stop_before_pixels=True)
            planes[plane] = PlaneInput(
                images=outcome.sample.images,
                image_orientation_patient=[float(v) for v in header.ImageOrientationPatient],
                pixel_spacing=[float(v) for v in header.PixelSpacing],
            )

        features = build_study_features(
            planes,
            _study_laterality(series_df, series_root, study_id, sink=sink),
            encode_batch,
            IMAGE_MEAN,
            IMAGE_STD,
            embedding_dim=WIDE_EMBEDDING_DIM,
            slice_pool=slice_pool,
        )
        if not features.laterality_reliable:
            sink["laterality_unreliable_studies"] += 1
        if not features.has_any_plane:
            sink["studies_with_no_plane"] += 1
        if timings is not None:
            timings.append(
                {
                    "slices": study_slices,
                    "seconds": time.perf_counter() - study_start,
                }
            )
        if per_plane is not None:
            per_plane.append(features.plane_embeddings)
        # Slice the reported baseline back out of the wide vector: the CLS
        # half plus the four flags, exactly the frozen 388-wide contract.
        vectors.append(
            np.concatenate(
                [
                    features.vector[:pooled_width],
                    features.vector[WIDE_EMBEDDING_DIM:],
                ]
            )
        )
    return np.vstack(vectors)

import pydicom  # noqa: E402  (imported after the source path is established)

labeled_features = pd.DataFrame(
    build_features_for(
        train_series_df, DATA_DIR / "train_series", labeled_studies["StudyInstanceUID"]
    )
)
y = labeled_studies[LABEL_COLUMNS].astype(int).reset_index(drop=True)
selected_splits, folds = select_multilabel_folds(y, seed=SEED)
cv_result = cross_validate_image_model(labeled_features, y, folds)

# The comparison is against the reported baseline, so that baseline must be
# the one on record rather than something re-derived to resemble it.
EXPECTED_BASELINE_MACRO_AUC = 0.6345688959
if abs(cv_result.pooled_macro_auc - EXPECTED_BASELINE_MACRO_AUC) > 1e-9:
    raise RuntimeError("baseline macro AUC does not reproduce")

baseline_summary = pd.Series(
    {
        "Evaluation studies": float(len(labeled_studies)),
        "Baseline pooled OOF macro AUC": cv_result.pooled_macro_auc,
        "Fold assignment signature": fold_signature(
            labeled_studies["StudyInstanceUID"].tolist(), folds
        ),
    },
    name="Value",
).to_frame()

display(baseline_summary)

**Interpretation:** the baseline's out-of-fold predictions are the reference the
weak-trained model is compared against, and it must reproduce the score already
on record before any comparison is drawn from it.

## 4. Weak-Labelled Training Set

Report-only studies, disjoint from the evaluation set by construction. The
count is capped for runtime, not for any statistical reason — extraction costs
roughly two and a half seconds per study, and the cap keeps a multi-hour run
comfortably inside the budget while still supplying vastly more studies than
the human-labelled set.

In [ ]:
# Capped for budget, and sampled deterministically so the training set is a
# property of the seed rather than of row order on the day.
WEAK_TRAINING_STUDIES = 3000

# Sorted first, then permuted from a fixed seed, so the selection depends on
# the seed rather than on the order rows happen to arrive in.
weak_pool = unlabeled_studies.sort_values("StudyInstanceUID").reset_index(drop=True)
permutation = np.random.default_rng(SEED).permutation(len(weak_pool))
weak_studies = weak_pool.iloc[
    permutation[:WEAK_TRAINING_STUDIES]
].reset_index(drop=True)

# Disjointness is the property the whole evaluation rests on, so it is checked
# rather than trusted to the upstream split.
if set(weak_studies["StudyInstanceUID"]) & set(labeled_studies["StudyInstanceUID"]):
    raise RuntimeError("weak training set overlaps the evaluation set")

weak_labels = weak_label_frame(weak_studies["Report"].tolist())

weak_sink = {key: (0 if not isinstance(v, list) else []) for key, v in telemetry.items()}
weak_start = time.perf_counter()
weak_features = pd.DataFrame(
    build_features_for(
        train_series_df,
        DATA_DIR / "train_series",
        weak_studies["StudyInstanceUID"],
        sink=weak_sink,
    )
)
weak_extraction_seconds = time.perf_counter() - weak_start

weak_label_support = pd.DataFrame(
    {
        "Resolved": weak_labels.notna().sum(),
        "Positive": (weak_labels == 1).sum(),
        "Negative": (weak_labels == 0).sum(),
        "Abstain rate": weak_labels.isna().mean(),
    }
).reindex(LABEL_COLUMNS)

display(weak_label_support)

**Interpretation:** the resolved counts are the point of the exercise — where the
human-labelled set offers ten to twenty examples per label, the reports resolve
hundreds or thousands. The abstain rate is the honest cost: a report that does
not speak to a finding contributes nothing for that label rather than a
fabricated negative.

## 5. Weak-Trained Heads Evaluated on the Human Labels

In [ ]:
# One head per label, fitted only on the rows where that label resolved. The
# evaluation studies are never fitted on, so these predictions are fully
# out-of-sample rather than merely out-of-fold.
weak_fit = fit_weak_label_heads(weak_features, weak_labels, labeled_features)
weak_probabilities = weak_fit.probabilities.set_index(y.index)

weak_delta = paired_bootstrap_delta(
    y, weak_probabilities, cv_result.oof_probabilities,
    iterations=BOOTSTRAP_ITERATIONS, seed=SEED,
)

weak_summary = pd.DataFrame(
    {
        "Baseline (58 human labels)": {
            "Training studies": float(len(labeled_studies)),
            "Macro AUC": cv_result.pooled_macro_auc,
            "Delta": 0.0,
            "Delta 95% lower": float("nan"),
            "Delta 95% upper": float("nan"),
            "Resolved": False,
        },
        "Weak labels (reports)": {
            "Training studies": float(len(weak_studies)),
            "Macro AUC": macro_auc(y, weak_probabilities),
            "Delta": weak_delta.delta,
            "Delta 95% lower": weak_delta.lower,
            "Delta 95% upper": weak_delta.upper,
            "Resolved": weak_delta.excludes_zero,
        },
    }
).T

display(weak_summary)

**Interpretation:** the registered rule is that weak-label training counts as an
improvement only if this interval excludes zero in its favour; a higher point
estimate does not suffice. Labels that could not be fitted contribute a constant
column, which scores exactly chance and so can neither help nor flatter the
macro average.

## 6. Per-Label Result and Persisted Summary

In [ ]:
weak_per_label = pd.DataFrame(
    {
        "Baseline": pd.Series(cv_result.pooled_per_label_auc),
        "Weak-trained": pd.Series(
            {
                label: macro_auc(y[[label]], weak_probabilities[[label]])
                for label in LABEL_COLUMNS
            }
        ),
        "Weak training rows": pd.Series(weak_fit.support),
        "Weak positives": pd.Series(weak_fit.positives),
    }
).reindex(LABEL_COLUMNS)
weak_per_label["Change"] = weak_per_label["Weak-trained"] - weak_per_label["Baseline"]

display(weak_per_label)

weak_run_summary = pd.Series(
    {
        "Weak training studies": float(len(weak_studies)),
        "Labels abstained": float(len(weak_fit.abstained)),
        "Weak extraction seconds": weak_extraction_seconds,
        "Seconds per weak study": weak_extraction_seconds / max(len(weak_studies), 1),
        "Report-only studies available": float(len(weak_pool)),
    },
    name="Value",
).to_frame()

display(weak_run_summary)

with open("/kaggle/working/weak_label_summary.json", "w") as handle:
    json.dump(
        {
            "baseline": json.loads(baseline_summary.to_json()),
            "weak_label_support": json.loads(weak_label_support.to_json()),
            "comparison": json.loads(weak_summary.to_json()),
            "per_label": json.loads(weak_per_label.to_json()),
            "run": json.loads(weak_run_summary.to_json()),
            "abstained_labels": list(weak_fit.abstained),
        },
        handle,
        indent=2,
    )

**Interpretation:** the per-label view is where a data lever should show itself.
A label whose weak training rows number in the hundreds and whose score does not
move is evidence that the weak label encodes a different event than the human
one, rather than a noisier version of it. Seconds per study says whether the
full report-only pool is affordable in a later run.